# Transformación de IMFs (CEEMDAN) a grafo mediante Horizontal Visibility Graph (HVG)

Este notebook replica el flujo de `042_imf_to_graph_hvg_transform.ipynb` usando las IMFs obtenidas con CEEMDAN (`data/20abr26/msci_world_imfs_ceemdan.parquet`).

El archivo de IMFs no incluye fechas en el índice; se alinean las fechas con `data/20abr26/msci_world.parquet` por posición de fila (misma longitud) para poder filtrar y visualizar en el tiempo.


In [1]:
import pandas as pd
import numpy as np
import torch
from ts2vg import HorizontalVG
from torch_geometric.data import Data


/home/agusriscos/proyectos/ARPTools/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Rutas de datos (CEEMDAN + fechas alineadas)
archivo_imfs = "data/20abr26/msci_world_imfs_ceemdan.parquet"
archivo_fechas = "data/20abr26/msci_world.parquet"

df_imfs = pd.read_parquet(archivo_imfs, engine="pyarrow")
df_precios = pd.read_parquet(archivo_fechas, engine="pyarrow")

if len(df_imfs) != len(df_precios):
    raise ValueError(
        f"Longitudes distintas: IMFs {len(df_imfs)} vs precios {len(df_precios)}. "
        "Revise que ambos datasets correspondan al mismo muestreo."
    )

df_imfs = df_imfs.copy()
df_imfs.index = df_precios.index

print(f"IMFs cargadas desde: {archivo_imfs}")
print(f"Fechas tomadas del índice de: {archivo_fechas}")
print(f"Shape del DataFrame de IMFs: {df_imfs.shape}")
print(f"\nColumnas disponibles: {list(df_imfs.columns)}")
print(f"\nRango de fechas disponible:")
print(f"  - Desde: {df_imfs.index.min()}")
print(f"  - Hasta: {df_imfs.index.max()}")


IMFs cargadas desde: data/20abr26/msci_world_imfs_ceemdan.parquet
Fechas tomadas del índice de: data/20abr26/msci_world.parquet
Shape del DataFrame de IMFs: (3587, 9)

Columnas disponibles: ['IMF_1', 'IMF_2', 'IMF_3', 'IMF_4', 'IMF_5', 'IMF_6', 'IMF_7', 'IMF_8', 'Residuo']

Rango de fechas disponible:
  - Desde: 2012-01-12 00:00:00-05:00
  - Hasta: 2026-04-20 00:00:00-04:00


In [3]:
# IMF a transformar (cambiar si se desea otra componente)
nombre_columna_imf = "IMF_1"

# ts2vg requiere un array escribible (no solo lectura desde parquet)
serie_imf = df_imfs[nombre_columna_imf].to_numpy(dtype=np.float64, copy=True)
print(f"Columna seleccionada: {nombre_columna_imf}")
print(f"Shape de la serie: {serie_imf.shape}")
print(f"Primeros 10 valores: {serie_imf[:10]}")


Columna seleccionada: IMF_1
Shape de la serie: (3587,)
Primeros 10 valores: [-0.12892771 -0.26184484  0.20892933 -0.12897592  0.31827241 -0.13278786
  0.07455606  0.03973479  0.09805686 -0.12520762]


In [4]:
# Construir el grafo HVG (Horizontal Visibility Graph)
hvg = HorizontalVG(directed="left_to_right")
grafo_hvg = hvg.build(serie_imf)

nodos = np.arange(len(serie_imf))
enlaces = np.array(grafo_hvg.edges)

print(f"Número de nodos: {len(nodos)}")
print(f"Número de enlaces: {len(enlaces)}")
print(f"\nPrimeros 10 nodos: {nodos[:10]}")
print(f"\nPrimeros 10 enlaces:")
print(enlaces[:10])
print(f"\nForma de los enlaces: {enlaces.shape}")


Número de nodos: 3587
Número de enlaces: 7161

Primeros 10 nodos: [0 1 2 3 4 5 6 7 8 9]

Primeros 10 enlaces:
[[3328 3329]
 [3327 3329]
 [3325 3329]
 [3324 3329]
 [2054 3329]
 [3329 3330]
 [3329 3331]
 [3329 3332]
 [3329 3333]
 [3329 3335]]

Forma de los enlaces: (7161, 2)


In [5]:
# Convertir el grafo HVG a un objeto Data de PyTorch Geometric
edge_index = torch.tensor(enlaces.T, dtype=torch.long)
node_features = torch.tensor(serie_imf, dtype=torch.float).unsqueeze(1)
data = Data(x=node_features, edge_index=edge_index)

print("Objeto Data creado:")
print(f"  - Número de nodos: {data.num_nodes}")
print(f"  - Número de enlaces: {data.num_edges}")
print(f"  - Features de nodos shape: {data.x.shape}")
print(f"  - Edge index shape: {data.edge_index.shape}")
print("\nPrimeros 5 features de nodos:")
print(data.x[:5])
print("\nPrimeros 10 enlaces (edge_index):")
print(data.edge_index[:, :10])


Objeto Data creado:
  - Número de nodos: 3587
  - Número de enlaces: 7161
  - Features de nodos shape: torch.Size([3587, 1])
  - Edge index shape: torch.Size([2, 7161])

Primeros 5 features de nodos:
tensor([[-0.1289],
        [-0.2618],
        [ 0.2089],
        [-0.1290],
        [ 0.3183]])

Primeros 10 enlaces (edge_index):
tensor([[3328, 3327, 3325, 3324, 2054, 3329, 3329, 3329, 3329, 3329],
        [3329, 3329, 3329, 3329, 3329, 3330, 3331, 3332, 3333, 3335]])


In [6]:
# Filtrar el grafo para el rango 2020-2025 (visualización Plotly)
import plotly.graph_objects as go

indice_original = df_imfs.index.copy()

if not isinstance(indice_original, pd.DatetimeIndex):
    print("Convirtiendo índice a DatetimeIndex...")
    indice_original = pd.to_datetime(indice_original)

try:
    if hasattr(indice_original, "tz") and indice_original.tz is not None:
        print("Normalizando timezone del índice...")
        indice_original = indice_original.tz_convert("UTC").tz_localize(None)
except Exception as e:
    print(f"Advertencia al normalizar timezone: {e}")
    try:
        indice_original = pd.to_datetime(indice_original).tz_localize(None)
    except Exception:
        pass

fecha_inicio = pd.Timestamp("2020-01-01")
fecha_fin = pd.Timestamp("2025-12-31")

if hasattr(fecha_inicio, "tz") and fecha_inicio.tz is not None:
    fecha_inicio = fecha_inicio.tz_localize(None)
if hasattr(fecha_fin, "tz") and fecha_fin.tz is not None:
    fecha_fin = fecha_fin.tz_localize(None)

mascara_fechas = (indice_original >= fecha_inicio) & (indice_original <= fecha_fin)
indices_filtrados = np.where(mascara_fechas)[0]

print(f"Filtrando grafo para el rango {fecha_inicio.date()} a {fecha_fin.date()}")
print(f"Nodos en el rango: {len(indices_filtrados)}")
if len(indices_filtrados) > 0:
    print(f"Índices: {indices_filtrados[0]} a {indices_filtrados[-1]}")
else:
    print("⚠ No se encontraron nodos en el rango especificado")
    print(f"Rango disponible: {indice_original.min()} a {indice_original.max()}")

nodos_validos = set(indices_filtrados)

edge_index_filtrado = data.edge_index.cpu().numpy()
enlaces_filtrados = []
for k in range(edge_index_filtrado.shape[1]):
    src = int(edge_index_filtrado[0, k])
    dst = int(edge_index_filtrado[1, k])
    if src in nodos_validos and dst in nodos_validos:
        enlaces_filtrados.append([src, dst])

if len(enlaces_filtrados) > 0:
    enlaces_filtrados = np.array(enlaces_filtrados, dtype=np.int64)
else:
    enlaces_filtrados = np.array([], dtype=np.int64).reshape(0, 2)
print(f"Enlaces en el subgrafo: {len(enlaces_filtrados)}")

mapeo_indices = {int(idx_original): int(idx_local) for idx_local, idx_original in enumerate(indices_filtrados)}
if len(enlaces_filtrados) > 0:
    enlaces_mapeados = np.array(
        [[mapeo_indices[int(src)], mapeo_indices[int(dst)]] for src, dst in enlaces_filtrados],
        dtype=np.int64,
    )
else:
    enlaces_mapeados = np.array([], dtype=np.int64).reshape(0, 2)

valores_imf_filtrados = serie_imf[indices_filtrados]
fechas_filtradas = indice_original[indices_filtrados]

print("\nDatos del subgrafo:")
print(f"  - Nodos: {len(indices_filtrados)}")
print(f"  - Enlaces: {len(enlaces_mapeados)}")
if len(fechas_filtradas) > 0:
    fecha_min = fechas_filtradas[0]
    fecha_max = fechas_filtradas[-1]
    if isinstance(fecha_min, pd.Timestamp):
        print(f"  - Rango de fechas: {fecha_min.date()} a {fecha_max.date()}")
    else:
        print(f"  - Rango de fechas: {fecha_min} a {fecha_max}")


Normalizando timezone del índice...
Filtrando grafo para el rango 2020-01-01 a 2025-12-31
Nodos en el rango: 1507
Índices: 2005 a 3511
Enlaces en el subgrafo: 2996

Datos del subgrafo:
  - Nodos: 1507
  - Enlaces: 2996
  - Rango de fechas: 2020-01-02 a 2025-12-30


In [7]:
# Visualización del subgrafo (español): layout de fuerzas + serie temporal
import networkx as nx
from plotly.subplots import make_subplots

if len(indices_filtrados) == 0:
    raise ValueError("No hay nodos en el rango de fechas especificado. Ajusta el rango de fechas.")

if len(enlaces_mapeados) == 0:
    print("⚠ Advertencia: No hay enlaces en el subgrafo filtrado.")

valores_imf_array = np.array(valores_imf_filtrados, dtype=np.float64)

print("Creando grafo NetworkX...")
G = nx.Graph()
G.add_nodes_from(range(len(indices_filtrados)))
if len(enlaces_mapeados) > 0:
    enlaces_tuplas = [(int(edge[0]), int(edge[1])) for edge in enlaces_mapeados]
    G.add_edges_from(enlaces_tuplas)
    print(f"Grafo creado con {G.number_of_nodes()} nodos y {G.number_of_edges()} enlaces")

print("Calculando layout del grafo...")
pos = nx.spring_layout(G, k=1, iterations=50, seed=42)

x_pos = np.array([pos[i][0] for i in range(len(indices_filtrados))])
y_pos = np.array([pos[i][1] for i in range(len(indices_filtrados))])


def formatear_fecha(fecha):
    """Formatea una fecha de forma segura."""
    try:
        if isinstance(fecha, pd.Timestamp):
            return fecha.strftime("%Y-%m-%d")
        if isinstance(fecha, str):
            return fecha
        return str(pd.Timestamp(fecha).strftime("%Y-%m-%d"))
    except Exception:
        return str(fecha)


textos_hover = [
    f"Fecha: {formatear_fecha(fecha)}<br>Valor {nombre_columna_imf}: {val:.4f}<br>Índice: {idx}"
    for fecha, val, idx in zip(fechas_filtradas, valores_imf_array, indices_filtrados)
]

print("Creando trazas de enlaces...")
edge_x = []
edge_y = []
if len(enlaces_mapeados) > 0:
    for edge in enlaces_mapeados:
        src_idx = int(edge[0])
        dst_idx = int(edge[1])
        x0, y0 = pos[src_idx]
        x1, y1 = pos[dst_idx]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode="lines",
    line=dict(width=0.5, color="#888"),
    hoverinfo="none",
    showlegend=False,
)

valor_min = np.min(valores_imf_array)
valor_max = np.max(valores_imf_array)

print("Creando traza de nodos...")
node_trace = go.Scatter(
    x=x_pos,
    y=y_pos,
    mode="markers",
    marker=dict(
        size=8,
        color=valores_imf_array,
        colorscale="Viridis",
        cmin=valor_min,
        cmax=valor_max,
        showscale=False,
        line=dict(width=1, color="black"),
    ),
    text=textos_hover,
    hoverinfo="text",
    showlegend=False,
)

print("Creando traza de serie temporal...")
serie_temporal_trace = go.Scatter(
    x=fechas_filtradas,
    y=valores_imf_array,
    mode="lines+markers",
    name=nombre_columna_imf,
    line=dict(color="rgba(128,128,128,0.3)", width=1),
    marker=dict(
        size=4,
        color=valores_imf_array,
        colorscale="Viridis",
        cmin=valor_min,
        cmax=valor_max,
        showscale=True,
        colorbar=dict(title=f"Valor {nombre_columna_imf}", x=1.02, len=0.4, y=0.25),
        line=dict(width=0.5, color="white"),
    ),
    hovertemplate="Fecha: %{x}<br>Valor " + nombre_columna_imf + ": %{y:.4f}<extra></extra>",
)

print("Creando figura con subplots...")
fig = make_subplots(
    rows=2,
    cols=1,
    row_heights=[0.6, 0.4],
    subplot_titles=(
        f"Grafo HVG - CEEMDAN {nombre_columna_imf} ({fecha_inicio.date()} a {fecha_fin.date()})",
        f"Serie temporal {nombre_columna_imf}",
    ),
    vertical_spacing=0.12,
)

fig.add_trace(edge_trace, row=1, col=1)
fig.add_trace(node_trace, row=1, col=1)
fig.add_trace(serie_temporal_trace, row=2, col=1)

fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=1)
fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=1)

fig.update_xaxes(title_text="Fecha", row=2, col=1)
fig.update_yaxes(title_text=f"Valor {nombre_columna_imf}", row=2, col=1)

fig.update_layout(
    title=dict(
        text=f"Grafo HVG y serie temporal - CEEMDAN {nombre_columna_imf} ({fecha_inicio.date()} a {fecha_fin.date()})",
        x=0.5,
        xanchor="center",
        font=dict(size=18),
    ),
    showlegend=False,
    hovermode="closest",
    height=900,
    margin=dict(b=50, l=50, r=50, t=80),
    annotations=[
        dict(
            text=f"Nodos: {len(indices_filtrados)} | Enlaces: {len(enlaces_mapeados)}",
            showarrow=False,
            xref="paper",
            yref="paper",
            x=0.005,
            y=0.48,
            xanchor="left",
            yanchor="bottom",
            font=dict(size=12),
        )
    ],
    plot_bgcolor="white",
)

archivo_html = "data/20abr26/grafo_hvg_ceemdan_2020_2025.html"
print(f"Guardando visualización en {archivo_html}...")
fig.write_html(archivo_html)
print(f"✓ Visualización guardada exitosamente en: {archivo_html}")

fig.show()


Creando grafo NetworkX...
Grafo creado con 1507 nodos y 2996 enlaces
Calculando layout del grafo...
Creando trazas de enlaces...
Creando traza de nodos...
Creando traza de serie temporal...
Creando figura con subplots...
Guardando visualización en data/20abr26/grafo_hvg_ceemdan_2020_2025.html...
✓ Visualización guardada exitosamente en: data/20abr26/grafo_hvg_ceemdan_2020_2025.html


In [8]:
# Graph visualization (English): force layout + time series
import networkx as nx
from plotly.subplots import make_subplots

if len(indices_filtrados) == 0:
    raise ValueError("No nodes in the specified date range. Adjust the date range.")

if len(enlaces_mapeados) == 0:
    print("⚠ Warning: No edges in the filtered subgraph.")

valores_imf_array = np.array(valores_imf_filtrados, dtype=np.float64)

print("Creating NetworkX graph...")
G = nx.Graph()
G.add_nodes_from(range(len(indices_filtrados)))
if len(enlaces_mapeados) > 0:
    enlaces_tuplas = [(int(edge[0]), int(edge[1])) for edge in enlaces_mapeados]
    G.add_edges_from(enlaces_tuplas)
    print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

print("Calculating graph layout...")
pos = nx.spring_layout(G, k=1, iterations=50, seed=42)

x_pos = np.array([pos[i][0] for i in range(len(indices_filtrados))])
y_pos = np.array([pos[i][1] for i in range(len(indices_filtrados))])


def format_date(fecha):
    """Formats a date safely."""
    try:
        if isinstance(fecha, pd.Timestamp):
            return fecha.strftime("%Y-%m-%d")
        if isinstance(fecha, str):
            return fecha
        return str(pd.Timestamp(fecha).strftime("%Y-%m-%d"))
    except Exception:
        return str(fecha)


hover_texts = [
    f"Date: {format_date(fecha)}<br>{nombre_columna_imf} value: {val:.4f}<br>Index: {idx}"
    for fecha, val, idx in zip(fechas_filtradas, valores_imf_array, indices_filtrados)
]

print("Creating edge traces...")
edge_x = []
edge_y = []
if len(enlaces_mapeados) > 0:
    for edge in enlaces_mapeados:
        src_idx = int(edge[0])
        dst_idx = int(edge[1])
        x0, y0 = pos[src_idx]
        x1, y1 = pos[dst_idx]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode="lines",
    line=dict(width=0.5, color="#888"),
    hoverinfo="none",
    showlegend=False,
)

valor_min = np.min(valores_imf_array)
valor_max = np.max(valores_imf_array)

print("Creating node trace...")
node_trace = go.Scatter(
    x=x_pos,
    y=y_pos,
    mode="markers",
    marker=dict(
        size=8,
        color=valores_imf_array,
        colorscale="Viridis",
        cmin=valor_min,
        cmax=valor_max,
        showscale=False,
        line=dict(width=1, color="black"),
    ),
    text=hover_texts,
    hoverinfo="text",
    showlegend=False,
)

print("Creating time series trace...")
time_series_trace = go.Scatter(
    x=fechas_filtradas,
    y=valores_imf_array,
    mode="lines+markers",
    name=nombre_columna_imf,
    line=dict(color="rgba(128,128,128,0.3)", width=1),
    marker=dict(
        size=4,
        color=valores_imf_array,
        colorscale="Viridis",
        cmin=valor_min,
        cmax=valor_max,
        showscale=True,
        colorbar=dict(title=f"{nombre_columna_imf} value", x=1.02, len=0.4, y=0.25),
        line=dict(width=0.5, color="white"),
    ),
    hovertemplate="Date: %{x}<br>" + nombre_columna_imf + ": %{y:.4f}<extra></extra>",
)

print("Creating figure with subplots...")
fig = make_subplots(
    rows=2,
    cols=1,
    row_heights=[0.6, 0.4],
    vertical_spacing=0.12,
)

fig.add_trace(edge_trace, row=1, col=1)
fig.add_trace(node_trace, row=1, col=1)
fig.add_trace(time_series_trace, row=2, col=1)

fig.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=1)
fig.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, row=1, col=1)

fig.update_xaxes(title_text="Date", row=2, col=1)
fig.update_yaxes(title_text=f"{nombre_columna_imf} value", row=2, col=1)

fig.update_layout(
    showlegend=False,
    hovermode="closest",
    height=900,
    margin=dict(b=50, l=50, r=50, t=50),
    plot_bgcolor="white",
)

archivo_html = "data/20abr26/grafo_hvg_ceemdan_2020_2025_english.html"
print(f"Saving visualization to {archivo_html}...")
fig.write_html(archivo_html)
print(f"✓ Visualization saved successfully to: {archivo_html}")

fig.show()


Creating NetworkX graph...
Graph created with 1507 nodes and 2996 edges
Calculating graph layout...
Creating edge traces...
Creating node trace...
Creating time series trace...
Creating figure with subplots...
Saving visualization to data/20abr26/grafo_hvg_ceemdan_2020_2025_english.html...
✓ Visualization saved successfully to: data/20abr26/grafo_hvg_ceemdan_2020_2025_english.html
